# Let's go PRO!

Advanced RAG Techniques!

Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [ ]:
from pathlib import Path
import os
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth, helpers
import boto3
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go


load_dotenv(override=True)

MODEL = "gpt-4.1-nano"

DB_NAME = "preprocessed_db"
collection_name = "docs"
OPENSEARCH_INDEX = "preprocessed-db-docs"
OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
OPENSEARCH_SERVICE = os.getenv("OPENSEARCH_SERVICE", "aoss")  # use "es" for a managed OpenSearch domain
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

openai = OpenAI()

if not OPENSEARCH_HOST:
    raise ValueError("Set OPENSEARCH_HOST in your .env file to your AWS OpenSearch endpoint (without https://).")

credentials = boto3.Session().get_credentials()
auth = AWSV4SignerAuth(credentials, AWS_REGION, OPENSEARCH_SERVICE)

opensearch = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST.replace("https://", "").rstrip("/") , "port": 443}],
    http_auth=auth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    pool_maxsize=20,
)

In [2]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [3]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in AWS OpenSearch

That's it!

### Let's start with Step 1


In [4]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [5]:
documents = fetch_documents()

Loaded 76 documents


### Donezo! On to Step 2 - make the chunks

In [6]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [7]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: company
The document has been retrieved from: knowledge-base/company/about.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 5 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# About Insurellm

Insurellm was founded by Avery La

In [8]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [9]:
make_messages(documents[0])

[{'role': 'user',
  'content': "\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: company\nThe document has been retrieved from: knowledge-base/company/about.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 5 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the document:\n\n# A

In [10]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [11]:
process_document(documents[0])

[Result(page_content='Introduction to Insurellm\n\nInsurellm is an insurance technology company founded in 2015 by Avery Lancaster. It started with a marketplace connecting consumers and providers and has since expanded into multiple insurance product platforms, focusing on innovation and growth.\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.', metadata={'source': 'knowledge-base/company/about.md', 'type': 'company'}),
 Result(page_content='Growth and Expansion (2015-2020)\n\nBetween its founding and 2020, Insurellm experienced rapid expansion, developing various insurance portals and reaching a peak of 200 employees across 12 offices in the US.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home

In [12]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [13]:
chunks = create_chunks(documents)

100%|██████████| 76/76 [08:26<00:00,  6.66s/it]


In [14]:
print(len(chunks))

432


### Finally, Step 3 - save the embeddings to AWS OpenSearch

The notebook uses AWS SigV4 authentication. For OpenSearch Serverless set `OPENSEARCH_SERVICE=aoss`; for a managed OpenSearch domain use `OPENSEARCH_SERVICE=es`.


In [ ]:
def create_embeddings(chunks):
    # Recreate the OpenSearch vector index so this notebook run starts clean.
    if opensearch.indices.exists(index=OPENSEARCH_INDEX):
        opensearch.indices.delete(index=OPENSEARCH_INDEX)

    dimension = len(openai.embeddings.create(model=embedding_model, input=["dimension check"]).data[0].embedding)

    opensearch.indices.create(
        index=OPENSEARCH_INDEX,
        body={
            "settings": {
                "index": {
                    "knn": True
                }
            },
            "mappings": {
                "properties": {
                    "embedding": {
                        "type": "knn_vector",
                        "dimension": dimension,
                        "space_type": "cosinesimil"
                    },
                    "text": {"type": "text"},
                    "source": {"type": "keyword"},
                    "type": {"type": "keyword"}
                }
            }
        }
    )

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    actions = []
    for i, (chunk, vector) in enumerate(zip(chunks, vectors)):
        actions.append({
            "_index": OPENSEARCH_INDEX,
            "_id": str(i),
            "_source": {
                "embedding": vector,
                "text": chunk.page_content,
                "source": chunk.metadata["source"],
                "type": chunk.metadata["type"],
            }
        })

    helpers.bulk(opensearch, actions)
    opensearch.indices.refresh(index=OPENSEARCH_INDEX)
    count = opensearch.count(index=OPENSEARCH_INDEX)["count"]
    print(f"OpenSearch vector index created with {count} documents")


In [ ]:
create_embeddings(chunks)

# Nothing more to do here... right?

Wait! Didja think I'd forget??

In [ ]:
result = opensearch.search(
    index=OPENSEARCH_INDEX,
    body={
        "size": 10000,
        "query": {"match_all": {}},
        "_source": ["embedding", "text", "source", "type"]
    }
)
hits = result["hits"]["hits"]
vectors = np.array([hit["_source"]["embedding"] for hit in hits])
documents = [hit["_source"]["text"] for hit in hits]
metadatas = [{"source": hit["_source"]["source"], "type": hit["_source"]["type"]} for hit in hits]
doc_types = [metadata["type"] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]


In [ ]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D AWS OpenSearch Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D AWS OpenSearch Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

## And now - let's build an Advanced RAG!

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing

In [20]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [21]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [ ]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = opensearch.search(
        index=OPENSEARCH_INDEX,
        body={
            "size": RETRIEVAL_K,
            "query": {
                "knn": {
                    "embedding": {
                        "vector": query,
                        "k": RETRIEVAL_K
                    }
                }
            },
            "_source": ["text", "source", "type"]
        }
    )

    chunks = []
    for hit in results["hits"]["hits"]:
        source = hit["_source"]
        chunks.append(
            Result(
                page_content=source["text"],
                metadata={"source": source["source"], "type": source["type"]}
            )
        )
    return chunks


In [ ]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [24]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

Promotion to Se...
Additional HR N...
Annual Performa...
Performance and...
Performance rat...
Career Progress...
Annual Performa...
Performance His...
Insurellm Caree...
Performance His...


In [25]:
reranked = rerank(question, chunks)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [27]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

Promotion to Se...
Additional HR N...
Annual Performa...
Performance and...
Performance rat...
Career Progress...
Annual Performa...
Performance His...
Insurellm Caree...
Performance His...


In [29]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

In [28]:
reranked = rerank(question, chunks)

[9, 12, 11, 7, 3, 17, 4, 13, 5, 1, 2, 6, 8, 10, 14, 15, 16, 18, 19, 20]


In [30]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

In [31]:
reranked[0].page_content

'Career Progression at Insurellm and Previous Roles (Part 1)\n\nMichelle Rivera has been working at Insurellm as a Product Designer since February 2019, leading design for the Rellm reinsurance platform. Prior to this, she worked at DesignStudio NYC and CreativeAgency in various design roles.\n\n## Insurellm Career Progression\n- **February 2019 - Present:** Product Designer\n  - Leads design for Rellm reinsurance platform\n  - Creates user experiences for complex B2B workflows\n  - Collaborates with product managers and engineers\n  - Conducts user research with enterprise clients\n\n- **August 2016 - January 2019:** Senior UX/UI Designer at DesignStudio NYC\n  - Designed digital products for various clients\n  - Specialized in enterprise software and complex workflows'

In [32]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [33]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [34]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [ ]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [ ]:
rewrite_query("Who won the IIOTY award?", [])

In [ ]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [44]:
answer_question("Who won the IIOTY award?", [])

Who received the IIOTY award?
[1, 2, 18, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20]


('Maxine Thompson, a Senior Data Engineer at Insurellm, received the IIOTY (Insurellm Innovator of the Year) award in 2023.',
 [Result(page_content='Promotion to Senior Data Engineer and Achievements (2021 - present)\n\nIn 2021, Maxine became Senior Data Engineer, leading impactful projects, mentoring, and receiving recognition such as the 2023 IIOTY award.\n\n- **January 2021 - Present**: **Senior Data Engineer**  \n  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued asset at Insurellm. She was recognized as Insurellm Innovator of the year in 2023, receiving the prestigious IIOTY 2023 award.', metadata={'source': 'knowledge-base/employees/Maxine Thompson.md', 'type': 'employees'}),
  Result(page_content="Additional HR Notes and Initiatives\n\nMaxine's HR record includes participation in 

In [49]:
answer_question("Who went to University of Manchester?", [])

Who attended the University of Manchester?
[1, 8, 18, 2, 3, 13, 14, 4, 5, 6, 7, 9, 10, 11, 12, 15, 16, 17, 19, 20]


('I do not have any information indicating that any of the employees at Insurellm attended the University of Manchester.',
 [Result(page_content="Educational background and board participation\n\nJames Wilson holds advanced degrees in computer science and engineering, serves on advisory boards, and has received industry recognition.\n\n## Other HR Notes\n- **Education:** MS in Computer Science from MIT, BS in Computer Engineering from Stanford University\n- **Board Participation:** Serves on technical advisory board for two InsurTech startups\n- **Industry Recognition:** Named to InsureTech 100 list (2021, 2022, 2023), speaker at major technology conferences\n- **Patents:** 5 patents in distributed systems and insurance technology\n- **Leadership:** Known for building high-performing teams and fostering innovation culture. Strong mentor to technical leaders.\n- **Strategic Impact:** Transformed Insurellm's technology from legacy systems to modern cloud-native architecture. Key architec